# Notebook 04 — Text / NLP, Shipping Network Graph & Association Rules

**Member 4 | Lectures 9, 10, 18**

This notebook covers three major analytics tasks on the Olist e-commerce dataset:

| Section | Lecture | Topic |
| :--- | :--- | :--- |
| A | Lecture 9 | Text Analytics — clean Portuguese reviews, TF-IDF, top complaint terms |
| B | Lecture 10 | Shipping Network Graph — 27-state directed graph, centrality, communities |
| C | Lecture 18 | Association Rule Mining — category co-purchases & bad-review risk rules |

All data is read from `PROCESSED_DATA/` (generated by Notebook 01).

---
## 0. Setup & Imports

In [ ]:
# ── Standard libraries ──────────────────────────────────────────────
import os
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# ── Visualization ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# ── NLP / Text (Lecture 9) ──────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Graph Analysis (Lecture 10) ─────────────────────────────────────
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

# ── Association Rules (Lecture 18) ──────────────────────────────────
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

print('All imports successful ✓')

### 0.1  Load Processed Data

We load four CSV files from `PROCESSED_DATA/`:
- **`master_orders.csv`** — 1 row per order with delivery metrics, review scores, seller/customer states, lateness buckets.
- **`cleaned_reviews.csv`** — De-duplicated reviews with `has_comment` flag.
- **`cleaned_items.csv`** — Order-item level data with English category names.
- **`master_sellers.csv`** — 1 row per seller with aggregated performance metrics.

In [ ]:
# ── Path to processed data ──────────────────────────────────────────
DATA_DIR = os.path.join('..', 'PROCESSED_DATA')

# ── Load datasets ───────────────────────────────────────────────────
orders   = pd.read_csv(os.path.join(DATA_DIR, 'master_orders.csv'))
reviews  = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_reviews.csv'))
items    = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_items.csv'))
sellers  = pd.read_csv(os.path.join(DATA_DIR, 'master_sellers.csv'))

print(f'master_orders  : {orders.shape[0]:>8,} rows × {orders.shape[1]} cols')
print(f'cleaned_reviews: {reviews.shape[0]:>8,} rows × {reviews.shape[1]} cols')
print(f'cleaned_items  : {items.shape[0]:>8,} rows × {items.shape[1]} cols')
print(f'master_sellers : {sellers.shape[0]:>8,} rows × {sellers.shape[1]} cols')

In [ ]:
# Quick peek at the reviews table (our main input for Section A)
reviews.head(3)

In [ ]:
# Quick peek at the orders table (used for graph & rules)
orders[['order_id', 'seller_state', 'customer_state',
         'days_late', 'is_late', 'lateness_bucket', 'review_score']].head(3)

In [ ]:
# Check how many reviews actually have text comments
n_total   = len(reviews)
n_comment = int(reviews['has_comment'].sum())
pct       = n_comment / n_total * 100
print(f'Reviews with text: {n_comment:,} / {n_total:,}  ({pct:.1f}%)')
print(f'Reviews without text: {n_total - n_comment:,}  ({100 - pct:.1f}%)')